In [ ]:
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer


In [ ]:

# Download necessary NLTK resources
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:

# Initialize stemmer and lemmatizer
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()



In [ ]:

# Load dataset
path = "/content/Resume-final-v3.csv"

In [ ]:

df = pd.read_csv(path).dropna()  # Remove empty rows


In [ ]:
######################################################
if 'index' not in df.columns:
    df.insert(0, 'index', range(1, len(df) + 1))
else:
    print("Column 'index' already exists in the DataFrame.")
##############################################################


In [ ]:
# Convert text to lowercase
df['Resume'] = df['Resume'].str.lower()
df['Category']=df['Category'].str.lower()


In [ ]:


# Remove duplicates
df.drop_duplicates(inplace=True)


In [ ]:

# Define stop words
stop_words = set(stopwords.words('english'))


In [ ]:

# Text preprocessing function (before tokenization)
def preprocess_text(text):
    if isinstance(text, str):
        text = re.sub(r'[^a-z\s]', '', text)  # Remove special characters, keep letters and spaces
        text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
        text = re.sub('http\S+\s*', ' ', text)  # remove URLs
        text = re.sub('RT|cc', ' ', text)  # remove RT and cc
        text = re.sub('#\S+', '', text)  # remove hashtags
        text = re.sub('@\S+', '  ', text)  # remove mentions
        text = re.sub('[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), ' ', text)
# remove punctuations
        text = re.sub(r'[^\x00-\x7f]',r' ', text)
        text = re.sub('\s+', ' ', text)  # remove extra whitespace

        return text
    return ""


In [ ]:

# Apply preprocessing to the 'Resume' column
df['Cleaned_Resume'] = df['Resume'].apply(preprocess_text)


In [ ]:

# Tokenize AFTER cleaning
df['tokens'] = df['Cleaned_Resume'].apply(word_tokenize)


In [ ]:

# Remove stopwords, apply stemming and lemmatization to tokens
def process_tokens(tokens):
    tokens = [word for word in tokens if word not in stop_words]  # Remove stopwords
    tokens = [stemmer.stem(word) for word in tokens]  # Apply stemming
    tokens = [lemmatizer.lemmatize(word) for word in tokens]  # Apply lemmatization
    return tokens


In [ ]:

df['tokens'] = df['tokens'].apply(process_tokens)



In [ ]:
df.head()

,index,Category,Resume,Cleaned_Resume,tokens
0,1,data science,"areas of interest deep learning, control syste...",areas of interest deep learning control system...,"[area, interest, deep, learn, control, system,..."
1,2,data science,skills â¢ r â¢ python â¢ sap hana â¢ table...,skills r python sap hana tableau sap hana sql ...,"[skill, r, python, sap, hana, tableau, sap, ha..."
2,3,data science,"education details \n mca ymcaust, faridabad...",education details mca ymcaust faridabad haryan...,"[educ, detail, mca, ymcaust, faridabad, haryan..."
3,4,data science,"skills c basics, iot, python, matlab, data sci...",skills c basics iot python matlab data science...,"[skill, c, basic, iot, python, matlab, data, s..."
4,5,data science,skills â¢ python â¢ tableau â¢ data visuali...,skills python tableau data visualization r stu...,"[skill, python, tableau, data, visual, r, stud..."


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,accuracy_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
import joblib
import pickle

In [ ]:

df.head()


,index,Category,Resume,Cleaned_Resume,tokens
0,1,data science,"areas of interest deep learning, control syste...",areas of interest deep learning control system...,"[area, interest, deep, learn, control, system,..."
1,2,data science,skills â¢ r â¢ python â¢ sap hana â¢ table...,skills r python sap hana tableau sap hana sql ...,"[skill, r, python, sap, hana, tableau, sap, ha..."
2,3,data science,"education details \n mca ymcaust, faridabad...",education details mca ymcaust faridabad haryan...,"[educ, detail, mca, ymcaust, faridabad, haryan..."
3,4,data science,"skills c basics, iot, python, matlab, data sci...",skills c basics iot python matlab data science...,"[skill, c, basic, iot, python, matlab, data, s..."
4,5,data science,skills â¢ python â¢ tableau â¢ data visuali...,skills python tableau data visualization r stu...,"[skill, python, tableau, data, visual, r, stud..."


In [ ]:

# Replace NaN values with empty strings in the 'Resume' column
df['tokens'] = df['tokens'].fillna('')

In [ ]:

 #Drop rows with NaN in 'Category' column
df.dropna(subset=['Category'], inplace=True)


In [ ]:

# Assumes `Resume` column contains resume text and `Category` column contains target labels
resumes = df['tokens']
categories = df['Category']


In [ ]:

category_counts = df['Category'].value_counts()
filtered_category_counts = category_counts[category_counts >= 2]
print(filtered_category_counts)


Category
sales                        268
hr                           259
information-technology       238
business-development         236
fitness                      233
consultant                   230
healthcare                   226
designer                     212
teacher                      204
advocate                     200
digital-media                190
arts                         129
chef                         128
agriculture                  126
aviation                     117
accountant                   116
engineering                  116
finance                      116
banking                      115
construction                 111
public-relations             110
apparel                       96
java developer                84
testing                       70
automobile                    68
devops engineer               55
python developer              48
web designing                 43
hadoop                        42
bpo                           42
b

In [ ]:

filtered_categories = filtered_category_counts.index
print(filtered_categories)


Index(['sales', 'hr', 'information-technology', 'business-development',
       'fitness', 'consultant', 'healthcare', 'designer', 'teacher',
       'advocate', 'digital-media', 'arts', 'chef', 'agriculture', 'aviation',
       'accountant', 'engineering', 'finance', 'banking', 'construction',
       'public-relations', 'apparel', 'java developer', 'testing',
       'automobile', 'devops engineer', 'python developer', 'web designing',
       'hadoop', 'bpo', 'blockchain', 'operations manager',
       'mechanical engineer', 'etl developer', 'database', 'pmo',
       'health and fitness', 'electrical engineering', 'data science',
       'business analyst', 'dotnet developer', 'automation testing',
       'network security engineer', 'sap developer', 'civil engineer'],
      dtype='object', name='Category')


In [ ]:

# Step 6: Encode Target Labels
le = LabelEncoder()

In [ ]:

# Filter the DataFrame to include only the filtered categories
df_filtered = df[df['Category'].isin(filtered_categories)]


In [ ]:

# Now, fit_transform on the 'Category' column of the filtered DataFrame
df_filtered['Category_encoded'] = le.fit_transform(df_filtered['Category'])
y = df_filtered['Category_encoded']  # Encoded labels


<ipython-input-25-9bb5b6305009>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['Category_encoded'] = le.fit_transform(df_filtered['Category'])


In [ ]:

# Save the LabelEncoder
#joblib.dump(le, 'xgboost_LabelEncoder.pkl')


In [ ]:

# Step 2: Feature Extraction using TF-IDF
tfidf = TfidfVectorizer(sublinear_tf=True,
    stop_words='english',
    max_features=1500)  # You can adjust max_features


In [ ]:


# Filter the resumes based on the filtered categories
filtered_resumes = df[df['Category'].isin(filtered_categories)]['tokens']


In [ ]:

# Join the tokens back into strings before passing to TF-IDF
filtered_resumes = filtered_resumes.apply(lambda tokens: ' '.join(tokens))


In [ ]:

X = tfidf.fit_transform(filtered_resumes).toarray()


In [ ]:

# Save the Vectorizer
#joblib.dump(tfidf, 'xgboost_Model_Vectorizer.pkl')


In [ ]:

# Step 7: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


In [ ]:

# Model Training and Testing
#xgb_model = xgb.XGBClassifier()
#xgb_model.fit(X_train, y_train)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# Define classifiers
classifiers = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(),
    "Support Vector Machine": SVC(),
    "XGBoost": XGBClassifier()
}

# List to store results
classifier_results = []

# Train and evaluate
for name, model in classifiers.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro')
    rec = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')

    classifier_results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1
    })


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:

# Convert to DataFrame
clf_df = pd.DataFrame(classifier_results)

# Save to Excel
clf_df.to_excel('classification_results.xlsx', index=False)

print("Classification results saved to 'classification_results.xlsx'")


Classification results saved to 'classification_results.xlsx'


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pandas as pd

# Define regressors
regressors = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(),
    "Support Vector Regressor": SVR(),
    "XGBoost Regressor": XGBRegressor()
}

# Store results
regressor_results = []

# Train and evaluate
for name, model in regressors.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    rmse = mse ** 0.5
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    regressor_results.append({
        'Model': name,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2 Score': r2
    })


In [ ]:
# Convert to DataFrame
reg_df = pd.DataFrame(regressor_results)

# Save to Excel
reg_df.to_excel('regression_metrics.xlsx', index=False)

print("Regression metrics saved to 'regression_metrics.xlsx'")


Regression metrics saved to 'regression_metrics.xlsx'


In [ ]:

# Predict on the training data:
y_train_pred = xgb_model.predict(X_train)


In [ ]:
# Calculate training accuracy:
train_accuracy = accuracy_score(y_train, y_train_pred)
print("Training Accuracy Score:", train_accuracy)


In [ ]:
#joblib.dump(xgb_model, 'xgb_model.pkl')

In [ ]:

#Save the Trained Model
#joblib.dump(xgb_model, 'xgboost_Model.pkl')
#model = pickle.load(open('xgb_model.pkl', 'rb'))
#model.get_booster().save_model("xgb_model.json")
#model.save_model('xgboost_model.json')


In [ ]:
#Save the Trained Model
#joblib.dump(xgb_model, 'xgboost_Model.pkl')

#booster = model.get_booster()            # if you have a sklearn wrapper
xgb_model.save_model("resume_ranker.xgb")
#xgb_model.save_model('xgboost_model.json') # Changed model to xgb_model

In [ ]:

# Calculate and print the accuracy score
xgb_prediction = xgb_model.predict(X_test)  # Predict on the test data
accuracy = accuracy_score(y_test, xgb_prediction)
print("Testing Accuracy Score:", accuracy)


In [ ]:

# Predict on the testing data to get the predicted labels
y_pred = xgb_model.predict(X_train)


In [ ]:

# Use the actual true labels from the testing set (y_test)
y_true = y_train


In [ ]:

# Generate the confusion matrix
cm = confusion_matrix(y_true, y_pred)
# Display the confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
fig, ax = plt.subplots(figsize=(15,12))
disp.plot(ax=ax, cmap=plt.cm.Blues)


# Add labels and title
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.title("Training Confusion Matrix")

# Show the plot
plt.show()


In [ ]:


# Predict on the testing data to get the predicted labels
y_pred = xgb_model.predict(X_test)

# Use the actual true labels from the testing set (y_test)
y_true = y_test

# Generate the confusion matrix
cm = confusion_matrix(y_true, y_pred)
# Display the confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
fig, ax = plt.subplots(figsize=(15,12))
disp.plot(ax=ax, cmap=plt.cm.Blues)


# Add labels and title
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.title("Testing Confusion Matrix")

# Show the plot
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Count only the categories that actually appear (non-zero count)
category_counts = df['Category'].value_counts()
category_counts = category_counts[category_counts > 5].reset_index()
category_counts.columns = ['Category', 'ResumeCount']

# Create the bar chart
plt.figure(figsize=(10, 6))
sns.barplot(data=category_counts, x='Category', y='ResumeCount', palette='viridis')

# Simplify the chart to only show category bars

plt.xlabel("Category")
plt.ylabel("Number of Resumes")
plt.title("Distribution of Resumes by Category")
plt.xticks(rotation=45, ha="right")  # Rotate x-axis labels for better readability
plt.tight_layout()
plt.show()


In [ ]:

# Step 9: Ranking Resumes within a Category
def rank_resumes(category, resumes, classifier, tfidf, label_encoder):
    # Get the label for the specified category
    category_label = label_encoder.transform([category])[0]
    # Filter resumes belonging to a category
    category_resumes = df[df['Category'] == category]['tokens']
    # Join the tokens back into strings before transforming
    category_resumes = category_resumes.apply(lambda tokens: ' '.join(tokens))
    category_features = tfidf.transform(category_resumes)

    # Generate scores based on probabilities for the given category
    scores = classifier.predict_proba(category_features)[:, category_label]

    # Rank resumes by scores
    ranked_indices = np.argsort(scores)[::-1]
    ranked_resumes = category_resumes.iloc[ranked_indices]
    return ranked_resumes


In [ ]:
# Example: Rank Resumes for a Specific Category
cat=input("Enter the category to rank (CAPITAL LETTERS): ")
category_to_rank = cat  # Replace with your desired category
ranked_resumes = rank_resumes(category_to_rank, df['tokens'], xgb_model, tfidf, le)


In [ ]:

# Display Top Ranked Resumes
quant=input("Enter the number of resumes to display (NUMERIC): ")
quant=int(quant)
print(f"Top-ranked resumes for category '{category_to_rank}':")
print(ranked_resumes[:quant])  # Display top 5 resumes